# Time Indexes

## Motivation

In the previous notebook we gave our synthetic series a `DatetimeIndex` without dwelling on why. Here we look at that index itself: what it is, why it matters, and how to work with it.

A pandas `Series` or `DataFrame` can carry a plain, meaningless `RangeIndex` (`0, 1, 2, ...`). For time series data, using the actual timestamps as the index instead unlocks pandas' time-aware behavior: label-based slicing by date, alignment of series on their timestamps, and operations like resampling and rolling windows that we'll cover in later notebooks. Without a proper time index, a time series is just an ordinary array that happens to be sorted; with one, pandas understands the temporal structure and lets you reason about the data in calendar terms.

## The DatetimeIndex

The most common time index is `pd.DatetimeIndex`, a sequence of `Timestamp` objects. The easiest way to build one is `pd.date_range`, which takes a `start`, an `end` or a `periods` count, and a `freq` (frequency) string describing the spacing between timestamps.

Some common frequency strings:

- `"D"` -- calendar day
- `"B"` -- business day
- `"W"` -- weekly
- `"MS"` / `"ME"` -- month start / month end
- `"h"` -- hourly
- `"min"` -- minutely

In [27]:
import numpy as np
import pandas as pd

# Build a DatetimeIndex directly
idx = pd.date_range(start="2025-01-01", periods=5, freq="D")
print(idx)
print("freq:", idx.freq)

# Use it as the index of a Series
values = np.arange(5)
series = pd.Series(values, index=idx, name="example")
series

DatetimeIndex(['2025-01-01', '2025-01-02', '2025-01-03', '2025-01-04',
               '2025-01-05'],
              dtype='datetime64[ns]', freq='D')
freq: <Day>


2025-01-01    0
2025-01-02    1
2025-01-03    2
2025-01-04    3
2025-01-05    4
Freq: D, Name: example, dtype: int64

## Other Index Flavors

`DatetimeIndex` represents specific instants in time, but pandas has two closely related index types worth knowing:

- **`PeriodIndex`** -- represents a *span* of time (e.g., "March 2024" or "Q1 2025") rather than an instant. Useful when data is naturally bucketed into periods, like monthly sales totals, and you want arithmetic that respects period boundaries.
- **`TimedeltaIndex`** -- represents *durations* or *elapsed time* rather than points in time, e.g., "3 days after the start of an experiment." Useful when data is indexed by elapsed time rather than calendar time.

In [28]:
# A span of time per entry, not an instant
period_idx = pd.period_range(start="2025-01", periods=4, freq="M")
print(period_idx)

# Elapsed time per entry, not a calendar date
timedelta_idx = pd.timedelta_range(start="0D", periods=5, freq="D")
print(timedelta_idx)

PeriodIndex(['2025-01', '2025-02', '2025-03', '2025-04'], dtype='period[M]')
TimedeltaIndex(['0 days', '1 days', '2 days', '3 days', '4 days'], dtype='timedelta64[ns]', freq='D')


## Indexing and Slicing by Date

Once a `Series` or `DataFrame` has a proper `DatetimeIndex`, you can select data using date strings directly, and pandas interprets them intelligently -- a behavior called **partial string indexing**. A partial date string (e.g., just a year or a year-month) is expanded to match every timestamp within that period.

We'll build a slightly longer example series to see this in action.

In [29]:
# A longer example series to slice
rng = np.random.default_rng(seed=0)
dates = pd.date_range(start="2024-01-01", periods=365, freq="D")
long_series = pd.Series(rng.normal(size=len(dates)), index=dates, name="value")

# Select an entire year
print(long_series["2024"].shape)

# Select an entire month
print(long_series["2024-06"].shape)

# Select a date range with .loc
long_series.loc["2024-03-01":"2024-03-07"]

(365,)
(30,)


2024-03-01   -0.436435
2024-03-02   -1.169802
2024-03-03    1.739368
2024-03-04   -0.495911
2024-03-05    0.328970
2024-03-06   -0.258573
2024-03-07    1.583473
Freq: D, Name: value, dtype: float64

## Real-World Examples

Real-world data makes time-index quirks much more concrete than synthetic data. The three series below each have a genuinely different indexing structure, and that difference has real modeling consequences. All three examples query live data sources, so a fixed historical window (2023-01-01 through 2024-06-30) is used throughout, both to keep the output stable and to make the series directly comparable.

### Stock Market Data -- Irregular Trading Calendar

Stock data isn't evenly spaced in wall-clock time. Markets are closed on weekends and holidays, so guessing at a schedule with `freq="B"` (business days) gets you close but not exact -- it doesn't know about Carnival, Tiradentes Day, or other exchange-specific holidays.

In [30]:
import yfinance as yf
import pandas_market_calendars as mcal

start_date = "2023-01-01"
end_date = "2024-06-30"

# Daily closing prices for Petrobras (PETR4) on B3, the Brazilian exchange
data = yf.download("PETR4.SA", start=start_date, end=end_date, progress=False)
close = data["Close"].squeeze().rename("close")
print("freq:", close.index.freq)  # None -- irregular, trading days only

# Compare a naive "every business day" guess against the real exchange calendar
b3 = mcal.get_calendar("BMF")  # B3 / Bovespa
trading_days = b3.schedule(start_date=start_date, end_date=end_date).index

naive_bdays = pd.bdate_range(start=start_date, end=end_date)
false_trading_days = naive_bdays.difference(trading_days)
print(f"{len(false_trading_days)} weekdays that 'B' assumes are trading days but aren't:")
print(false_trading_days[:8])

freq: None
18 weekdays that 'B' assumes are trading days but aren't:
DatetimeIndex(['2023-02-20', '2023-02-21', '2023-04-07', '2023-04-21',
               '2023-05-01', '2023-06-08', '2023-09-07', '2023-10-12'],
              dtype='datetime64[ns]', freq=None)


The mismatched dates are exactly the local holidays `freq="B"` can't know about. A mismatch between your data's index and a naive assumption like this is a real, common source of silent bugs -- e.g., forward-filling across a holiday you didn't know existed.

### CDI and IPCA -- via `python-bcb`

`python-bcb` wraps the Brazilian Central Bank's SGS (Sistema Gerenciador de Séries Temporais) web service and returns pandas objects directly, which is more convenient than parsing the raw API by hand. Two series from the same source can still have very different index semantics:

In [31]:
from bcb import sgs, currency

bcb_start = pd.to_datetime(start_date)
bcb_end = pd.to_datetime(end_date)

df_cdi = sgs.get(("CDI", 12), start=bcb_start, end=bcb_end)  # daily accrual rate
df_ipca = sgs.get(("IPCA", 433), start=bcb_start, end=bcb_end)  # monthly inflation

print("CDI: one row per business day --", len(df_cdi), "rows")
print(df_cdi.head())
print()
print("IPCA: one row per month, but stamped as a specific day --", len(df_ipca), "rows")
print(df_ipca.head())

CDI: one row per business day -- 373 rows
                 CDI
Date                
2023-01-02  0.050788
2023-01-03  0.050788
2023-01-04  0.050788
2023-01-05  0.050788
2023-01-06  0.050788

IPCA: one row per month, but stamped as a specific day -- 18 rows
            IPCA
Date            
2023-01-01  0.53
2023-02-01  0.84
2023-03-01  0.71
2023-04-01  0.61
2023-05-01  0.23


In [32]:
# Convert IPCA's DatetimeIndex to a proper PeriodIndex for calendar-correct arithmetic
df_ipca_period = df_ipca.copy()
df_ipca_period.index = df_ipca_period.index.to_period("M")
print(df_ipca_period.head())

# USD/BRL PTAX rate -- also business-day irregular, like the stock data above
df_usdbrl = currency.get("USD", start=bcb_start, end=bcb_end)
print()
print(df_usdbrl.head())

         IPCA
Date         
2023-01  0.53
2023-02  0.84
2023-03  0.71
2023-04  0.61
2023-05  0.23

               USD
Date              
2023-01-02  5.3436
2023-01-03  5.3759
2023-01-04  5.4459
2023-01-05  5.4026
2023-01-06  5.2855


A few index-related points from this comparison:

- **Both come back as a `DatetimeIndex`, not a `PeriodIndex` -- even IPCA.** Each IPCA row is stamped `YYYY-MM-01`, a specific day standing in for the whole month, rather than an explicit period. If you're going to do calendar-correct arithmetic on it (accumulate 12 months, shift by quarters, etc.), convert explicitly with `df_ipca.index.to_period("M")`.
- **CDI and IPCA are on different implicit frequencies.** CDI is published on every business day, so it has a dense, near-daily index; IPCA has one row per month at the same daily resolution. This mismatch is exactly why you can't blindly `pd.concat` or join them without resampling one to match the other first.
- **`currency.get("USD", ...)` inherits the same trading-day irregularity as the stock example above** -- it's PTAX data, published on business days only (and on BCB's own holiday calendar, which doesn't perfectly match B3's), so the same caution about not assuming `freq="B"` applies here too.

### SELIC -- Irregular by Design

SELIC (Brazil's benchmark policy rate) is a good example of a series that's *intentionally* irregular underneath a deceptively regular-looking index. BCB's API returns the SELIC target as a daily-indexed series, but the value itself only changes on COPOM meeting dates -- roughly every 45 days, and not on a fixed schedule. Most of those daily rows are just repeating the last decision, not new information.

In [33]:
selic_target = sgs.get(("SELIC_TARGET", 432), start=bcb_start, end=bcb_end)

# The real information is in the change points, not the daily padding
changed = selic_target["SELIC_TARGET"].diff().ne(0)
print(f"{len(selic_target)} daily rows, but only {changed.sum()} actual COPOM decisions:")
selic_target[changed]

547 daily rows, but only 8 actual COPOM decisions:


,SELIC_TARGET
Date,
2023-01-01,13.75
2023-08-03,13.25
2023-09-21,12.75
2023-11-02,12.25
2023-12-14,11.75
2024-02-01,11.25
2024-03-21,10.75
2024-05-09,10.50


Comparable series elsewhere: the US **Fed funds rate** (set by FOMC, ~8 meetings/year), the Eurozone's **ECB deposit facility rate**, and the UK's **Bank Rate** (set by the BoE's MPC). All share the same structural quirk -- a step function tied to decision dates, not a smooth daily process -- even though each central bank publishes it on a different cadence.

---

A general principle these examples illustrate together: **the "right" index type is a modeling decision, not just a data-loading detail.** Trading data wants a market-calendar-aware irregular index; IPCA wants period semantics; SELIC wants you to be deliberate about the gap between "decision dates" and "calendar days." Getting this wrong doesn't usually throw an error -- it just quietly biases your features (e.g., a naive `ffill` treating a stale rate as if it were fresh information).

One habit that pays off, especially with external data sources like this: always check `df.index.freq` and `df.index.is_monotonic_increasing` right after loading, and decide explicitly whether you want daily, business-day, or period semantics before merging series from different sources.

## What's Next

Now that we've seen how a time index represents and organizes timestamps, the next notebook digs into the mechanics of working with dates and times themselves in `pandas` -- parsing strings into timestamps, timezones, and date arithmetic.